# 08 — YOLOv8 evaluation, generalization analysis and fine-tuning

This notebook loads the detector trained in notebook 07 and evaluates it on
the **test** split, which was never used during training or validation.

It:

1. loads `models/yolo/brain_tumor_yolov8n.pt`;
2. computes precision, recall, mAP@0.5 and mAP@0.5:0.95 on the test set;
3. displays the Ultralytics plots (confusion matrix, PR and F1 curves);
4. compares ground truth and predictions on sample test images;
5. measures localization errors (TP / FP / FN at IoU 0.5);
6. compares validation and test scores to judge generalization;
7. optionally fine-tunes the model with a small learning rate;
8. saves the evaluation artefacts under `models/yolo/evaluation/`.

> **How to read this notebook:** every code cell is preceded by a short
> summary that explains what the step does and how to interpret its output.

## 1. Imports and paths

Imports Ultralytics, OpenCV for reading the test images, and the plotting
helpers used to draw boxes.

**How to read the output:** the paths of the trained weights, the dataset
configuration and the evaluation folder. If the weights are missing, run
notebook 07 first.

In [ ]:
from pathlib import Path
import json
import shutil

import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from IPython.display import Image as IPythonImage
from IPython.display import display as ipython_display
from ultralytics import YOLO

ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

DATA_YAML = ROOT_DIR / "data" / "yolo" / "data.yaml"
DATASET_DIR = ROOT_DIR / "data" / "yolo"
MODELS_DIR = ROOT_DIR / "models" / "yolo"
RUNS_DIR = MODELS_DIR / "runs"
RUN_NAME = "brain_tumor_yolov8n"
SAVED_MODEL_PATH = MODELS_DIR / "brain_tumor_yolov8n.pt"
TRAINING_SUMMARY_PATH = MODELS_DIR / "training_summary.json"
EVAL_DIR = MODELS_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

CONF_THRESHOLD = 0.25
IOU_THRESHOLD = 0.5
SEED = 42

print("Ultralytics:", ultralytics.__version__)
print("Weights    :", SAVED_MODEL_PATH)
print("Dataset    :", DATA_YAML)
print("Evaluation :", EVAL_DIR)

## 2. Load the trained detector

Loads the best checkpoint saved by notebook 07. If the copy is missing, the
notebook falls back to the `best.pt` of the last training run, so it still
works after a run that was not exported.

**How to read the output:** the model summary (layers, parameters, GFLOPs) and
the loaded path. The model is in inference mode; no training happens here
except in the optional fine-tuning section.

In [ ]:
weights = SAVED_MODEL_PATH
if not weights.exists():
    weights = RUNS_DIR / RUN_NAME / "weights" / "best.pt"

if not weights.exists():
    raise FileNotFoundError(
        "No trained model found. Run 07_yolo_training.ipynb first, or place "
        f"a checkpoint at {SAVED_MODEL_PATH}."
    )

training_summary = None
if TRAINING_SUMMARY_PATH.exists():
    with open(TRAINING_SUMMARY_PATH) as handle:
        training_summary = json.load(handle)
    EVAL_IMGSZ = int(training_summary.get("imgsz", 640))
    EVAL_BATCH = int(training_summary.get("batch", 8))
else:
    EVAL_IMGSZ, EVAL_BATCH = 640, 8
    print("training_summary.json not found — using imgsz=640, batch=8.")

model = YOLO(str(weights))
print("Loaded weights:", weights)
print("Evaluation size:", EVAL_IMGSZ, "| batch:", EVAL_BATCH)
model.info()

## 3. Evaluate on the test split

Runs `model.val(split="test")`. The test split contains 255 images that were
never seen during training or validation, so these scores measure
generalization, not memorization.

**How to read the output:** four numbers for the whole dataset:
`precision` (how many predicted boxes are correct), `recall` (how many real
objects are found), `mAP50` (mean average precision at IoU 0.5) and
`mAP50-95` (averaged over IoU 0.5 to 0.95, the strictest score).

In [ ]:
test_metrics = model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=EVAL_IMGSZ,
    batch=EVAL_BATCH,
    device=0 if torch.cuda.is_available() else "cpu",
    project=str(RUNS_DIR),
    name=f"{RUN_NAME}_test",
    exist_ok=True,
    plots=True,
)

print(json.dumps(test_metrics.results_dict, indent=2))

## 4. Per-class test metrics

Builds a table with precision, recall, AP@0.5 and AP@0.5:0.95 for each class on
the test split.

**How to read the output:** classes are sorted by AP@0.5:0.95. A class with
high recall but low precision produces too many false boxes; the opposite
(low recall, high precision) means it misses objects. Compare this table with
the validation table of notebook 07.

In [ ]:
def per_class_table(metrics):
    rows = []
    for index, class_id in enumerate(metrics.box.ap_class_index):
        rows.append(
            {
                "class": metrics.names[int(class_id)],
                "precision": float(metrics.box.p[index]),
                "recall": float(metrics.box.r[index]),
                "mAP50": float(metrics.box.ap50[index]),
                "mAP50-95": float(metrics.box.ap[index]),
            }
        )
    return pd.DataFrame(rows).sort_values("mAP50-95", ascending=False, ignore_index=True)


test_table = per_class_table(test_metrics)
display(test_table)

## 5. Display the Ultralytics evaluation plots

Shows the images written by `model.val` in the evaluation run directory:
confusion matrix, precision-recall curve and F1 curve.

**How to read the output:**

- the **confusion matrix** diagonal is correct detections; off-diagonal cells
  are confusions between two classes, and the background row/column shows
  false positives / missed objects;
- the **PR curve** shows the precision/recall trade-off: the closer it stays
  to the top-right corner, the better;
- the **F1 curve** shows the confidence threshold that maximizes the F1 score.

In [ ]:
for plot_name in [
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "BoxPR_curve.png",
    "BoxF1_curve.png",
]:
    plot_path = Path(test_metrics.save_dir) / plot_name
    if plot_path.exists():
        print(plot_name)
        ipython_display(IPythonImage(filename=str(plot_path), width=720))
    else:
        print("missing plot:", plot_path)

## 6. Compare ground truth and predictions on test images

Runs the detector on a few test images (one per class) with
`conf >= 0.25` and displays the annotation on the left and the prediction on
the right.

**How to read the output:** the left panel is the reference annotation, the
right panel is what the model found. A correct detection has a box at the same
place with the right class; a missing box is a false negative; an extra box is
a false positive.

In [ ]:
CLASS_COLORS = {0: "#e74c3c", 1: "#3498db", 2: "#2ecc71", 3: "#9b59b6"}


def read_label_boxes(label_path: Path):
    boxes = []
    if not label_path.exists():
        return boxes
    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue
        class_id, x, y, w, h = line.split()
        boxes.append((int(class_id), float(x), float(y), float(w), float(h)))
    return boxes


def draw_truth(axis, image, boxes):
    height, width = image.shape[:2]
    axis.imshow(image)
    axis.axis("off")
    for class_id, x, y, w, h in boxes:
        x1, y1 = (x - w / 2) * width, (y - h / 2) * height
        axis.add_patch(
            patches.Rectangle(
                (x1, y1), w * width, h * height,
                linewidth=2, edgecolor=CLASS_COLORS[class_id], facecolor="none",
            )
        )
        axis.text(
            x1, max(y1 - 4, 8), test_metrics.names[class_id],
            color="white", fontsize=8,
            bbox={"facecolor": CLASS_COLORS[class_id], "pad": 2, "edgecolor": "none"},
        )


def draw_prediction(axis, image, result):
    axis.imshow(image)
    axis.axis("off")
    boxes = result.boxes
    if boxes is None:
        return
    for xyxy, confidence, class_id in zip(
        boxes.xyxy.cpu().numpy(),
        boxes.conf.cpu().numpy(),
        boxes.cls.cpu().numpy().astype(int),
    ):
        x1, y1, x2, y2 = xyxy
        axis.add_patch(
            patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor=CLASS_COLORS[class_id], facecolor="none",
            )
        )
        axis.text(
            x1, max(y1 - 4, 8), f"{test_metrics.names[class_id]} {confidence:.2f}",
            color="white", fontsize=8,
            bbox={"facecolor": CLASS_COLORS[class_id], "pad": 2, "edgecolor": "none"},
        )


test_labels = sorted((DATASET_DIR / "labels" / "test").glob("*.txt"))
sample_labels = []
for class_id in range(len(test_metrics.names)):
    for label_path in test_labels:
        if any(box[0] == class_id for box in read_label_boxes(label_path)):
            sample_labels.append(label_path)
            break

sample_images = [DATASET_DIR / "images" / "test" / f"{label.stem}.jpg" for label in sample_labels]
predictions = model.predict(
    source=[str(path) for path in sample_images],
    conf=CONF_THRESHOLD,
    iou=IOU_THRESHOLD,
    imgsz=EVAL_IMGSZ,
    verbose=False,
)

fig, axes = plt.subplots(len(sample_images), 2, figsize=(9, 4 * len(sample_images)))
for row, (image_path, label_path, result) in enumerate(zip(sample_images, sample_labels, predictions)):
    image = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    draw_truth(axes[row, 0], image, read_label_boxes(label_path))
    axes[row, 0].set_title(f"Ground truth — {image_path.name}", fontsize=9)
    draw_prediction(axes[row, 1], image, result)
    axes[row, 1].set_title(f"Prediction — {len(result.boxes)} box(es)", fontsize=9)

plt.tight_layout()
plt.show()

## 7. Localization error analysis (TP / FP / FN at IoU 0.5)

For every test image, matches each predicted box to a ground-truth box of the
same class when their IoU is at least 0.5. Unmatched predictions are false
positives, unmatched annotations are false negatives.

**How to read the output:** the counts per class. A high number of false
negatives means missed tumors (dangerous in a medical context); false positives
mean extra boxes on healthy regions. The precision/recall computed here should
be close to the Ultralytics metrics of step 3 — a large difference would mean
the matching threshold is not the one used by the evaluation.

In [ ]:
def box_iou(box_a, box_b):
    xa, ya = max(box_a[0], box_b[0]), max(box_a[1], box_b[1])
    xb, yb = min(box_a[2], box_b[2]), min(box_a[3], box_b[3])
    intersection = max(0.0, xb - xa) * max(0.0, yb - ya)
    area_a = max(0.0, box_a[2] - box_a[0]) * max(0.0, box_a[3] - box_a[1])
    area_b = max(0.0, box_b[2] - box_b[0]) * max(0.0, box_b[3] - box_b[1])
    union = area_a + area_b - intersection
    return intersection / union if union > 0 else 0.0


counts = {class_id: {"tp": 0, "fp": 0, "fn": 0} for class_id in test_metrics.names}

all_test_images = sorted((DATASET_DIR / "images" / "test").glob("*.jpg"))
all_predictions = model.predict(
    source=[str(path) for path in all_test_images],
    conf=CONF_THRESHOLD,
    iou=IOU_THRESHOLD,
    imgsz=EVAL_IMGSZ,
    verbose=False,
)

for image_path, result in zip(all_test_images, all_predictions):
    image = cv2.imread(str(image_path))
    height, width = image.shape[:2]
    label_path = DATASET_DIR / "labels" / "test" / f"{image_path.stem}.txt"

    ground_truth = []
    for class_id, x, y, w, h in read_label_boxes(label_path):
        ground_truth.append(
            (
                class_id,
                np.array([
                    (x - w / 2) * width,
                    (y - h / 2) * height,
                    (x + w / 2) * width,
                    (y + h / 2) * height,
                ]),
            )
        )

    predicted = [
        (int(class_id), xyxy)
        for xyxy, class_id in zip(
            result.boxes.xyxy.cpu().numpy(),
            result.boxes.cls.cpu().numpy().astype(int),
        )
    ]

    matched_truth = set()
    for class_id, pred_box in predicted:
        best_index, best_iou = None, 0.0
        for index, (gt_class, gt_box) in enumerate(ground_truth):
            if index in matched_truth or gt_class != class_id:
                continue
            overlap = box_iou(pred_box, gt_box)
            if overlap > best_iou:
                best_index, best_iou = index, overlap
        if best_index is not None and best_iou >= IOU_THRESHOLD:
            matched_truth.add(best_index)
            counts[class_id]["tp"] += 1
        else:
            counts[class_id]["fp"] += 1

    for index, (gt_class, _) in enumerate(ground_truth):
        if index not in matched_truth:
            counts[gt_class]["fn"] += 1

error_rows = []
for class_id, values in counts.items():
    tp, fp, fn = values["tp"], values["fp"], values["fn"]
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    error_rows.append(
        {
            "class": test_metrics.names[class_id],
            "true_boxes": tp + fn,
            "predicted_boxes": tp + fp,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "precision@0.5": precision,
            "recall@0.5": recall,
        }
    )

error_table = pd.DataFrame(error_rows)
display(error_table)

## 8. Generalization: validation vs test

Compares the validation metrics stored by notebook 07 with the test metrics
computed above, and plots the per-class AP@0.5 of the two sets.

**How to read the output:** test scores slightly below validation scores are
normal. A large drop (for example more than 0.10 of mAP50) means the model
does not generalize: it has memorized the training/validation images, the test
split is too different, or the dataset is too small. The printed verdict
summarizes the gap, and the chart shows which classes lose the most.

In [ ]:
if training_summary is not None:
    validation_metrics = training_summary["validation_metrics"]
else:
    validation_metrics = {}
    print("training_summary.json not found — run notebook 07 to compare val/test.")

comparison_rows = []
for key, test_value in test_metrics.results_dict.items():
    comparison_rows.append(
        {
            "metric": key,
            "validation": validation_metrics.get(key),
            "test": float(test_value),
            "gap (test - val)": (
                float(test_value) - validation_metrics[key]
                if key in validation_metrics
                else None
            ),
        }
    )
comparison = pd.DataFrame(comparison_rows)
display(comparison)

key_metric = "metrics/mAP50(B)"
if key_metric in validation_metrics:
    gap = test_metrics.results_dict[key_metric] - validation_metrics[key_metric]
    print(f"mAP50 gap (test - validation): {gap:+.4f}")
    if gap >= -0.05:
        print("Verdict: good generalization — test and validation scores are close.")
    elif gap >= -0.15:
        print("Verdict: moderate gap — acceptable, but check the weakest classes.")
    else:
        print("Verdict: large gap — the model likely over-fits; add data or regularize.")

if training_summary is not None:
    validation_table = pd.DataFrame(training_summary["per_class_validation"]).set_index("class")
    test_by_class = test_table.set_index("class")
    classes = [name for name in validation_table.index if name in test_by_class.index]
    positions = np.arange(len(classes))
    width = 0.38

    fig, axis = plt.subplots(figsize=(9, 4))
    axis.bar(positions - width / 2, validation_table.loc[classes, "mAP50"], width, label="validation")
    axis.bar(positions + width / 2, test_by_class.loc[classes, "mAP50"], width, label="test")
    axis.set_xticks(positions)
    axis.set_xticklabels(classes)
    axis.set_ylabel("mAP@0.5")
    axis.set_title("Per-class mAP@0.5 — validation vs test")
    axis.legend()
    plt.tight_layout()
    plt.show()

## 9. Optional fine-tuning

If `RUN_FINETUNE` is set to `True`, this cell restarts from the trained weights,
freezes the first 10 layers and trains again with a **10x smaller** learning
rate on the training split. This is the usual second stage after the initial
training: the backbone is only nudged, not retrained from scratch.

It is disabled by default because it adds a full training run. Evaluation of
the fine-tuned model is done on the test split and saved as
`brain_tumor_yolov8n_finetuned.pt`.

**How to read the output:** when disabled, a single `skipped` message. When
enabled, the same progress bar as notebook 07 and the test metrics of the
fine-tuned model; compare them with step 3 to decide whether it helped.

In [ ]:
RUN_FINETUNE = False
FINETUNE_EPOCHS = 10

if RUN_FINETUNE:
    with open(DATA_YAML) as handle:
        data_config = yaml.safe_load(handle)

    fine_model = YOLO(str(SAVED_MODEL_PATH))
    fine_model.train(
        data=str(DATA_YAML),
        epochs=FINETUNE_EPOCHS,
        imgsz=EVAL_IMGSZ,
        batch=EVAL_BATCH,
        lr0=0.001,
        freeze=10,
        patience=5,
        device=0 if torch.cuda.is_available() else "cpu",
        project=str(RUNS_DIR),
        name=f"{RUN_NAME}_finetune",
        exist_ok=True,
        seed=SEED,
        plots=True,
    )
    fine_metrics = fine_model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=EVAL_IMGSZ,
        batch=EVAL_BATCH,
        device=0 if torch.cuda.is_available() else "cpu",
        project=str(RUNS_DIR),
        name=f"{RUN_NAME}_finetune_test",
        exist_ok=True,
    )
    print(json.dumps(fine_metrics.results_dict, indent=2))

    fine_run = Path(fine_model.trainer.save_dir)
    shutil.copy2(fine_run / "weights" / "best.pt", MODELS_DIR / "brain_tumor_yolov8n_finetuned.pt")
    print("Fine-tuned model saved:", MODELS_DIR / "brain_tumor_yolov8n_finetuned.pt")
else:
    print("Fine-tuning skipped (set RUN_FINETUNE = True to run it).")

## 10. Save the evaluation artefacts

Writes `models/yolo/evaluation/test_evaluation.json` with the test metrics,
the per-class table, the error analysis and the validation/test comparison,
then copies the test run's CSV and plots next to it.

**How to read the output:** the paths of the saved JSON and of the evaluation
folder. These files are what you attach to a report or a presentation; they
make the results reproducible without rerunning the evaluation.

In [ ]:
test_run_dir = Path(test_metrics.save_dir)
if (test_run_dir / "results.csv").exists():
    shutil.copy2(test_run_dir / "results.csv", EVAL_DIR / "test_results.csv")

plots_dir = EVAL_DIR / "plots"
plots_dir.mkdir(exist_ok=True)
for plot_name in [
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "BoxPR_curve.png",
    "BoxF1_curve.png",
    "val_batch0_pred.jpg",
    "val_batch0_labels.jpg",
]:
    plot_path = test_run_dir / plot_name
    if plot_path.exists():
        shutil.copy2(plot_path, plots_dir / plot_name)

evaluation_summary = {
    "created_by": "08_yolo_evaluation.ipynb",
    "weights": str(weights.relative_to(ROOT_DIR)),
    "test_run_dir": str(test_run_dir.relative_to(ROOT_DIR)),
    "confidence_threshold": CONF_THRESHOLD,
    "iou_threshold": IOU_THRESHOLD,
    "test_metrics": {key: float(value) for key, value in test_metrics.results_dict.items()},
    "validation_metrics": validation_metrics,
    "per_class_test": test_table.to_dict(orient="records"),
    "error_analysis": error_table.to_dict(orient="records"),
    "validation_vs_test": comparison.to_dict(orient="records"),
}

with open(EVAL_DIR / "test_evaluation.json", "w") as handle:
    json.dump(evaluation_summary, handle, indent=2)

print("Saved evaluation:", EVAL_DIR / "test_evaluation.json")
print("Saved plots     :", plots_dir)

## Done

The YOLOv8 pipeline is now split into four notebooks:

- **`05_yolo_data_overview.ipynb`** — structure, pairing, YOLO format and
  class checks, sample images with boxes, validated manifest;
- **`06_yolo_preprocessing.ipynb`** — train/valid/test organization, file
  copy, `data.yaml` and augmentation configuration;
- **`07_yolo_training.ipynb`** — YOLOv8 training, curves, validation and
  model saving;
- **`08_yolo_evaluation.ipynb`** — test metrics, prediction inspection,
  generalization analysis and optional fine-tuning.

The trained model is `models/yolo/brain_tumor_yolov8n.pt` and the evaluation
report is `models/yolo/evaluation/test_evaluation.json`.